# Traditional Sentiment Analysis

In [1]:
from transformers import AutoTokenizer, AutoModel, pipeline
import pandas as pd
import numpy as np
import torch

/vast.mnt/home/20236408/thesis_nlp_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vast.mnt/home/20236408/thesis_nlp_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
df = pd.read_csv("Data/df_zondertips.csv")
documents = df["Article"].to_list()

In [3]:
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

In [4]:
def classify_article(article):
    result = classifier(
        article[:1000],
        ["positive news", "negative news", "neutral news"]
    )
    scores = dict(zip(result["labels"], result["scores"]))
    ss = (scores["positive news"] - scores["negative news"]) / (
        scores["positive news"] + scores["negative news"] + 1e-8
    )
    label = "POS" if ss > 0.2 else "NEG" if ss < -0.2 else "NEU"
    return {"label": label, "ss": ss}

In [5]:
results = []

for idx, row in df.iterrows():
    article = str(row["Article"]) if pd.notna(row["Article"]) else ""
    if not article.strip():
        continue

    sentiment = classify_article(article)

    results.append({
        "id": idx,
        "Headline": row.get("Headline", ""),
        "Date": row.get("Date", pd.NaT),
        "Year": row.get("Year", np.nan),
        "Newspaper": row.get("Newspaper", "Unknown"),
        "sentiment_label": sentiment["label"],
        "sentiment_score": sentiment["ss"]
    })

    if len(results) % 1000 == 0:
        print(f"Processed {len(results)} articles")

df_results = pd.DataFrame(results)
df_results.to_csv("Data/uniform_sentiment_results.csv", index=False)
print("Saved to Data/uniform_sentiment_results.csv")

Processed 1000 articles
Processed 2000 articles
Processed 3000 articles
Processed 4000 articles
Processed 5000 articles
Processed 6000 articles
Processed 7000 articles
Processed 8000 articles
Processed 9000 articles
Processed 10000 articles
Processed 11000 articles
Processed 12000 articles
Processed 13000 articles
Processed 14000 articles
Processed 15000 articles
Processed 16000 articles
Processed 17000 articles
Processed 18000 articles
Processed 19000 articles
Processed 20000 articles
Processed 21000 articles
Processed 22000 articles
Processed 23000 articles
Processed 24000 articles
Processed 25000 articles
Saved to Data/uniform_sentiment_results.csv


In [8]:
import pandas as pd

df_results = pd.read_csv("Data/uniform_sentiment_results.csv")

year_summary = (
    df_results
    .groupby("Year")
    .agg(
        n_articles=("id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        std_sentiment=("sentiment_score", "std"),
        pos_share=("sentiment_label", lambda x: (x == "POS").mean()),
        neg_share=("sentiment_label", lambda x: (x == "NEG").mean()),
        neu_share=("sentiment_label", lambda x: (x == "NEU").mean())
    )
    .reset_index()
    .sort_values("Year")
)

In [9]:
year_newspaper_summary = (
    df_results
    .groupby(["Newspaper", "Year"])
    .agg(
        n_articles=("id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        pos_share=("sentiment_label", lambda x: (x == "POS").mean()),
        neg_share=("sentiment_label", lambda x: (x == "NEG").mean()),
        neu_share=("sentiment_label", lambda x: (x == "NEU").mean())
    )
    .reset_index()
    .sort_values(["Newspaper", "Year"])
)

In [10]:
import plotly.express as px

# ─────────────────────────────────────────────────────────────────────────────
# 3. SHARED Y-AXIS RANGE
# Fix: compute a common range from both datasets so plots are comparable
# ─────────────────────────────────────────────────────────────────────────────
all_scores = pd.concat([
    year_summary["mean_sentiment"],
    year_newspaper_summary["mean_sentiment"]
])
y_min = round(all_scores.min() - 0.02, 2)
y_max = round(all_scores.max() + 0.02, 2)

# ─────────────────────────────────────────────────────────────────────────────
# 4. PLOTS
# ─────────────────────────────────────────────────────────────────────────────

# Plot 1 — Overall yearly trend
fig1 = px.line(
    year_summary, x="Year", y="mean_sentiment",
    markers=True,
    labels={"mean_sentiment": "Mean sentiment score"}
)
fig1.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    xaxis=dict(dtick=1),
    width=900, height=400,
    margin=dict(l=50, r=30, t=30, b=50)
)
fig1.add_hline(y=0, line_dash="dash", line_color="grey")
fig1.write_image("Output/MeanSentiment.png")

# Plot 2 — By newspaper
fig2 = px.line(
    year_newspaper_summary[year_newspaper_summary["n_articles"] >= 20],
    x="Year", y="mean_sentiment", color="Newspaper",
    markers=True,
    labels={"mean_sentiment": "Mean sentiment score"}
)
fig2.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    xaxis=dict(dtick=1),
    width=900, height=400,
    margin=dict(l=50, r=150, t=30, b=50),
    legend=dict(x=1.02, y=0.5, xanchor='left', yanchor='middle')
)
fig2.add_hline(y=0, line_dash="dash", line_color="grey")
fig2.write_image("Output/SentimentEvolution.png")

# Plot 3 — Sentiment label shares
fig3 = px.line(
    year_summary.melt(
        id_vars=["Year"],
        value_vars=["neg_share", "pos_share", "neu_share"],
        var_name="Label", value_name="Share"
    ).replace({
        "neg_share": "Negative",
        "pos_share": "Positive",
        "neu_share": "Neutral"
    }),
    x="Year", y="Share", color="Label",
    markers=True,
    labels={"Share": "Share of articles"},
    color_discrete_map={
        "Negative": "#E06666",
        "Positive": "#93C47D",
        "Neutral":  "#6FA8DC"
    }
)
fig3.update_layout(
    yaxis_tickformat=".0%",
    xaxis=dict(dtick=1),
    width=900, height=400,
    margin=dict(l=50, r=150, t=30, b=50),
    legend=dict(x=1.02, y=0.5, xanchor='left', yanchor='middle')
)
fig3.write_image("Output/SentimentLabels.png")

# Plot 4 — Article volume heatmap
fig4 = px.imshow(
    year_newspaper_summary.pivot(
        index="Newspaper", columns="Year", values="n_articles"
    ).fillna(0),
    labels={"color": "Articles"}
)
fig4.update_layout(
    width=900, height=300,
    margin=dict(l=50, r=30, t=30, b=50)
)
fig4.write_image("Output/Volume.png")

# KeyBERT (Keyword extraction) + Aspect Sentiment Analysis

Drug Class Assignment Function

In [1]:
import pandas as pd
import re

def assign_drug_classes(article_text):
    """Assign article to drug classes based on your regex terms."""
    classes = {
        'Cannabis': r'cannabi',
        'Stimulants': r'cocaïn|meth.*?(amfe|fetamine)',
        'Hallucinogens': r'xtc|ecstasy',
        'Dissociatives': r'ketamin',
        'Depressants': r'GHB|lachgas',
        'Designer': r'3[- ]?mmc|3mmc|3-mmc',
        'Synthetic_Opioids': r'synthetische opioïden',
    }
    assigned = []
    text_lower = article_text.lower()
    for cls, pattern in classes.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            assigned.append(cls)
    return assigned if assigned else ['None']

Traditional Sentiment Analysis w/ mDeBERTa Zero-Shot

In [2]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

def traditional_sentiment(article, truncate=1000):
    """Document-level sentiment (baseline)."""
    text = article[:truncate]
    result = classifier(text, ["positive news", "negative news", "neutral news"])
    scores = dict(zip(result["labels"], result["scores"]))
    ss = (scores["positive news"] - scores["negative news"]) / (
        scores["positive news"] + scores["negative news"] + 1e-8
    )
    label = "POS" if ss > 0.2 else "NEG" if ss < -0.2 else "NEU"
    return {"label": label, "ss": ss}

/vast.mnt/home/20236408/thesis_nlp_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vast.mnt/home/20236408/thesis_nlp_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Process Dataset

In [3]:
df = pd.read_csv("Data/df_zondertips.csv")

results = []
for idx, row in df.iterrows():
    article = str(row["Article"]) if pd.notna(row["Article"]) else ""
    if not article.strip():
        continue
    
    # Traditional sentiment
    sent = traditional_sentiment(article)
    
    # Drug classes
    classes = assign_drug_classes(article)
    
    for cls in classes:
        results.append({
            "id": idx,
            "Year": row.get("Year"),
            "Newspaper": row.get("Newspaper", "Unknown"),
            "Drug_Class": cls,
            "sentiment_label": sent["label"],
            "sentiment_score": sent["ss"]
        })

df_sent = pd.DataFrame(results)
df_sent.head()

,id,Year,Newspaper,Drug_Class,sentiment_label,sentiment_score
0,0,2020,De Telegraaf,Stimulants,NEG,-0.315017
1,1,2020,De Telegraaf,None,NEG,-0.870956
2,2,2020,de Volkskrant,None,NEG,-0.442953
3,3,2020,De Telegraaf,None,NEG,-0.548178
4,4,2017,AD/Algemeen Dagblad,None,NEG,-0.921717


Yearly Summaries by Drug Class

In [4]:
year_class_summary = (
    df_sent.groupby(["Drug_Class", "Year"])
    .agg(
        n_articles=("id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        pos_share=("sentiment_label", lambda x: (x == "POS").mean()),
        neg_share=("sentiment_label", lambda x: (x == "NEG").mean())
    )
    .reset_index()
    .sort_values(["Drug_Class", "Year"])
)

Visualizations

In [5]:
import plotly.express as px

# Overall trend by drug class
fig1 = px.line(
    year_class_summary[year_class_summary["n_articles"] >= 10],
    x="Year", y="mean_sentiment", color="Drug_Class",
    title="Traditional Sentiment: Drug Coverage Over Time"
)
fig1.show()

# Stacked sentiment shares
fig2 = px.area(
    year_class_summary.melt(id_vars=["Drug_Class", "Year"], value_vars=["pos_share", "neg_share"]),
    x="Year", y="value", color="Drug_Class", facet_col="variable",
    title="POS/NEG Share by Drug Class"
)
fig2.show()

In [9]:
import pandas as pd
import numpy as np
import re
import torch
from transformers import pipeline
import plotly.express as px

# ── Load paragraph-level data ─────────────────────────────────────────────────
df = pd.read_csv("Data/paragraphs_processed.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year

# Drop very short paragraphs (likely headers/bylines, not content)
df = df[df["Paragraph"].str.len() >= 80].copy()
print(f"Paragraphs after length filter: {len(df)}")

Paragraphs after length filter: 278117


Drug class definitions

In [10]:
DRUG_CLASSES = {
    "Cannabis": [
        r"cannabi", r"wiet", r"marihuana", r"marijuana",
        r"hasj", r"hennep", r"coffeeshop", r"nederwiet"
    ],
    "Stimulants": [
        r"coca[iï]n", r"\bcoke\b", r"meth.{0,3}(amfe|fetamine)",
        r"metamfetamin", r"\bspeed\b", r"amfetamin", r"amphetamin"
    ],
    "Hallucinogens": [
        r"\bxtc\b", r"ecstasy", r"\bmdma\b",
        r"psilocybin", r"\bpaddo", r"psychedeli"
    ],
    "Dissociatives": [
        r"ketamin", r"lachgas", r"distikstofoxide", r"\bn2o\b"
    ],
    "Depressants": [
        r"\bghb\b", r"gammahydroxy", r"\bgbl\b", r"gammabutyrolacton"
    ],
    "Designer_Drugs": [
        r"3[- ]?mmc", r"mephedron", r"nieuwe psychoactieve", r"\bnps\b"
    ],
    "Opioids": [
        r"synthetische opio", r"fentanyl", r"tramadol",
        r"oxycodon", r"hero[iï]ne?", r"opio[iï]d", r"\bmorfine\b"
    ],
}

COMPILED = {
    cls: [re.compile(p, re.IGNORECASE) for p in patterns]
    for cls, patterns in DRUG_CLASSES.items()
}

def detect_classes(text):
    """Return list of drug classes detected in this paragraph."""
    found = []
    for cls, patterns in COMPILED.items():
        if any(p.search(text) for p in patterns):
            found.append(cls)
    return found

# ── Tag each paragraph with its drug class(es) ───────────────────────────────
print("Detecting drug classes in paragraphs...")
df["drug_classes"] = df["Paragraph"].apply(detect_classes)
df_drug = df[df["drug_classes"].map(len) > 0].copy()

# Explode: one row per (paragraph × drug class)
df_drug = df_drug.explode("drug_classes").rename(columns={"drug_classes": "Drug_Class"})
print(f"Drug-relevant paragraph-class rows: {len(df_drug)}")
print(df_drug["Drug_Class"].value_counts())

Detecting drug classes in paragraphs...
Drug-relevant paragraph-class rows: 24254
Drug_Class
Stimulants        9268
Cannabis          7164
Hallucinogens     3151
Dissociatives     2008
Opioids           1996
Depressants        508
Designer_Drugs     159
Name: count, dtype: int64


Group paragraphs back to article level for classification

In [11]:
# Per article × drug class, concatenate all matching paragraphs (up to 800 chars)
# no sentence splitting needed, paragraphs ARE the unit
def concat_paragraphs(texts, max_chars=800):
    combined = " ".join(texts)
    return combined[:max_chars]

df_contexts = (
    df_drug
    .sort_values(["Art_ID", "Par_ID"])
    .groupby(["Art_ID", "Drug_Class", "Year", "Newspaper"])
    .agg(
        context=("Paragraph", lambda x: concat_paragraphs(x.tolist())),
        n_paragraphs=("Paragraph", "count"),
        Headline=("Headline", "first"),
    )
    .reset_index()
)
print(f"\nArticle × Drug_Class observations: {len(df_contexts)}")


Article × Drug_Class observations: 13949


In [12]:
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=0 if torch.cuda.is_available() else -1
)

LABELS = ["positief nieuws", "negatief nieuws", "neutraal nieuws"]
BATCH_SIZE = 32

def classify_batch(texts, aspects):
    # Aspect framing: "Over [Drug_Class]: [paragraph context]"
    framed = [f"Over {asp}: {txt}" for asp, txt in zip(aspects, texts)]
    results = classifier(framed, candidate_labels=LABELS, batch_size=BATCH_SIZE)
    outputs = []
    for r in results:
        scores = dict(zip(r["labels"], r["scores"]))
        pos = scores["positief nieuws"]
        neg = scores["negatief nieuws"]
        ss = (pos - neg) / (pos + neg + 1e-8)
        label = "POS" if ss > 0.2 else "NEG" if ss < -0.2 else "NEU"
        outputs.append({"label": label, "ss": ss})
    return outputs

print("\nRunning sentiment classification...")
all_labels, all_scores = [], []

for i in range(0, len(df_contexts), BATCH_SIZE):
    batch = df_contexts.iloc[i:i + BATCH_SIZE]
    results = classify_batch(batch["context"].tolist(), batch["Drug_Class"].tolist())
    all_labels.extend([r["label"] for r in results])
    all_scores.extend([r["ss"] for r in results])
    if (i // BATCH_SIZE) % 50 == 0:
        print(f"  {min(i + BATCH_SIZE, len(df_contexts))}/{len(df_contexts)}")

df_contexts["sentiment_label"] = all_labels
df_contexts["sentiment_score"] = all_scores

# Checkpoint
df_contexts.to_csv("Data/aspect_sentiment_paragraphs.csv", index=False)
print("Saved to Data/aspect_sentiment_paragraphs.csv")


Running sentiment classification...
  32/13949
  1632/13949
  3232/13949
  4832/13949
  6432/13949
  8032/13949
  9632/13949
  11232/13949
  12832/13949
Saved to Data/aspect_sentiment_paragraphs.csv


In [13]:
import plotly.express as px
import pandas as pd

df_contexts = pd.read_csv("Data/aspect_sentiment_paragraphs.csv")

MIN_N = 10

group_year = (
    df_contexts
    .groupby(["Drug_Class", "Year"])
    .agg(
        n=("sentiment_score", "count"),
        mean_ss=("sentiment_score", "mean"),
        std_ss=("sentiment_score", "std"),
        pos_share=("sentiment_label", lambda x: (x == "POS").mean()),
        neg_share=("sentiment_label", lambda x: (x == "NEG").mean()),
        neu_share=("sentiment_label", lambda x: (x == "NEU").mean()),
    )
    .reset_index()
)

group_np_year = (
    df_contexts
    .groupby(["Drug_Class", "Newspaper", "Year"])
    .agg(
        n=("sentiment_score", "count"),
        mean_ss=("sentiment_score", "mean"),
        pos_share=("sentiment_label", lambda x: (x == "POS").mean()),
        neg_share=("sentiment_label", lambda x: (x == "NEG").mean()),
    )
    .reset_index()
)

In [14]:
# Fig 1 — Aspect-level sentiment per drug class over time
fig1 = px.line(
    group_year[group_year["n"] >= MIN_N],
    x="Year", y="mean_ss", color="Drug_Class",
    markers=True,
    labels={"mean_ss": "Mean sentiment score", "Drug_Class": "Drug class"}
)
fig1.add_hline(y=0, line_dash="dash", line_color="grey")
fig1.update_layout(
    width=900, height=450,
    margin=dict(l=50, r=150, t=30, b=50),
    xaxis=dict(dtick=1),
    legend=dict(x=1.02, y=0.5, xanchor='left', yanchor='middle')
)
fig1.write_image("Output/DrugClassMeanSentiment.png")

# Fig 2 — Heatmap drug class x year
pivot = group_year.pivot(index="Drug_Class", columns="Year", values="mean_ss")
fig2 = px.imshow(
    pivot,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    labels={"color": "Mean SS"}
)
fig2.update_layout(
    width=900, height=350,
    margin=dict(l=50, r=50, t=30, b=50)
)
fig2.write_image("Output/DrugClassHeatmap.png")

# Fig 3 — Mean sentiment per drug class by newspaper
fig3 = px.bar(
    group_np_year[group_np_year["n"] >= MIN_N],
    x="Newspaper", y="mean_ss", color="Drug_Class",
    barmode="group",
    labels={"mean_ss": "Mean sentiment score", "Drug_Class": "Drug class"}
)
fig3.add_hline(y=0, line_dash="dash", line_color="grey")
fig3.update_layout(
    width=900, height=450,
    margin=dict(l=50, r=150, t=30, b=50),
    legend=dict(x=1.02, y=0.5, xanchor='left', yanchor='middle')
)
fig3.write_image("Output/DrugClassByNewspaper.png")

# Fig 4 — Faceted: sentiment per drug class x newspaper over time
fig4 = px.line(
    group_np_year[group_np_year["n"] >= MIN_N],
    x="Year", y="mean_ss", color="Newspaper",
    facet_col="Drug_Class", facet_col_wrap=3,
    markers=True,
    labels={"mean_ss": "Mean sentiment score"}
)
fig4.update_layout(
    width=1100, height=700,
    margin=dict(l=50, r=50, t=50, b=50)
)
fig4.write_image("DrugClassFaceted.png")

print("All figures exported.")

All figures exported.
